# 🦾 Can We Predict When a Robot Will Fail?

**Engineering portfolio project — Robotics + Data Science + Predictive Maintenance**

### Research question
> Can condition-monitoring data from a robotic arm help identify failure conditions?

The project uses the public Kaggle **Robot Predictive Maintenance** dataset. I deliberately evaluate the models on later operating conditions that were not used during training, because random row-level splitting can leak information from highly similar time-series observations.


## 1. Why I chose this problem

I am interested in mechanical/aerospace engineering and have had exposure to medical robotics. I wanted to explore how measurements from a physical machine can be transformed into a useful engineering warning.

The goal is not to claim that machine learning can replace an engineer. The goal is to investigate whether sensor patterns contain useful information about robot condition.

## 2. Dataset
- Training observations: **235,854**
- Test conditions held out chronologically: **20240503_163963, 20240503_164435, 20240503_164675, 20240503_165189**
- Motors: **6**
- Failure observations: **17,063 (7.23%)**

Each motor file contains time, position, temperature, voltage and a failure label.

## 3. Exploratory findings

The failures are highly uneven. Some operating conditions contain no labeled failures, while others contain substantial failure periods. Failure rates also differ strongly by motor.

This immediately creates an important modeling question: **Will a model generalize to a new operating condition, or will it simply memorize patterns associated with the conditions it has already seen?**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('../data/processed/condition_summary.csv')
display(df.sort_values('failures', ascending=False))

## 4. Feature engineering

In addition to the raw sensor readings, I calculated short rolling-window statistics:

- rolling mean
- rolling standard deviation
- 20-sample change
- relative time within the run

These features represent recent machine behavior rather than a single instantaneous reading.

## 5. Validation strategy

**Chronological holdout:** the final four training conditions are kept completely outside the training set.

This is intentionally stricter than a random row split. Rows from a time series that are next to each other can be extremely similar, so randomly mixing them between training and testing can produce an overly optimistic estimate of performance.

## 6. Models

I compared two models:

1. **Logistic Regression** — interpretable baseline.
2. **Random Forest** — nonlinear model that can capture interactions between sensor measurements.

Because failures are relatively uncommon, I used class weighting and reported ROC-AUC and Average Precision rather than accuracy alone.

## 7. Results
| Model | ROC-AUC | Average Precision | Precision @ 0.5 | Recall @ 0.5 | F1 @ 0.5 |
|---|---:|---:|---:|---:|---:|
| Logistic Regression | 0.704 | 0.068 | 0.000 | 0.000 | 0.000 |
| Random Forest | 0.833 | 0.135 | 0.000 | 0.000 | 0.000 |

The results show that performance is substantially harder to maintain when the model is tested on later operating conditions. This is a useful engineering lesson: **generalization matters more than an impressive score from a random split.**

## 8. Engineering interpretation

The model uses motor identity together with sensor measurements and recent trends. This is important because failure behavior is not identical across the six motors.

However, predictive association is not the same as physical causation. A sensor may be useful for prediction without being the physical reason the motor fails.


## 9. Limitations and next steps

- The dataset represents an educational robot rather than a medical robot.
- The available labels describe observed failure states; they do not prove a specific physical failure mechanism.
- Generalization to unseen operating conditions remains challenging.
- A safety-critical maintenance system would require much more validation.

### Next version
A natural extension is an **early-warning model** that predicts whether failure will occur within a defined time horizon, using only measurements available before the failure event.

## Final reflection

> The most important result was not simply the model score. The project taught me that engineering data can contain strong patterns that disappear when a model encounters a new operating condition. That made validation strategy part of the engineering problem itself.